## Convert CSV EMG file to H5 EMG file

In [1]:
import os, sys

from emg_file_converter import *
from emg_signal_extractor import *
import h5py
import numpy as np
# import pandas as pd
import matplotlib.pyplot as plt

path = os.path.abspath('./../../Processing/')
if(path not in sys.path):
    sys.path.append(path)
import ProcessingPipeline as pp

ModuleNotFoundError: No module named 'ruptures'

# Converte CSV file to H5

In [ ]:
Used_channels_matrix = np.array([ [13, 6, 49, 11, 15, 7, 10, 8],
                           [58, 12, 4, 59, 57, 14, 5, 16],
                           
                           [30, 40, 26, 33, 20, 24, 23, 32],
                           [45, 17, 18, 42, 39, 28, 37, 41]])
Used_channels=np.hstack(Used_channels_matrix)

In [ ]:
folder_path = './record_examples/sub03/'

In [ ]:
convert_csv_to_h5_emg_file(folder_path, extract_emg=False, Used_channels= np.array([13, 6, 49, 11, 15, 7, 10, 8]))

# Read the H5 file

In [ ]:
#h5_file = h5py.File('/root/data/data/Hyser_diff_ExtensorFlexor_2/train/Subject01.hdf5', 'r')
#h5_file = h5py.File('/root/data/data/65HD_gestures_comparison2/train/25295225.hdf5', 'r')


In [ ]:
h5_file = h5py.File(folder_path + 'emg_file.hdf5', 'r')

In [ ]:
h5_file.keys()

In [ ]:
h5_file['session_1'].keys()

In [ ]:
h5_file['session_1']['gesture_1'].keys()

In [ ]:
h5_file['session_1']['gesture_1']['repetition_1']['emg'][:].shape

In [ ]:
h5_file['session_1']['gesture_0']['repetition_1']['emg'][:].shape

In [ ]:
params_ch=[i-1 for i in Used_channels]
#for i in range(64):
for i in params_ch:
    print(i)
    plt.plot(h5_file['session_1']['gesture_0']['repetition_1']['emg'][i,:])
    plt.show()

# Apply preprocessing steps and Get train, vall and test dataset

In [ ]:
 folder_path+'train/sub03.hdf5'

In [ ]:
diretorio_train = os.path.dirname(folder_path+'train/sub03.hdf5')

In [ ]:
def create_data(source, destination, session_list, repetitions_list, gestures_list, processing=None):
    database_path = os.path.dirname(destination)
    if not os.path.exists(database_path):
        os.makedirs(database_path)
        print(f'Creating the {database_path}.')
    else:
        print(f'Using the {database_path} path.')
        
    with h5py.File(source, 'r') as h5_file:
        with h5py.File(destination, 'w') as h5_processed_file:
            h5_processed_file.create_dataset('sessions', data= np.array(session_list))
            # Iterate over sessions in the original file
            for s in session_list:
                # Create a group for the new session in the processed file
                session_group = h5_processed_file.create_group(f'session_{s}')
                
                session_group.create_dataset('gestures', data=np.array(gestures_list))
                # Iterate over gestures in the current session
                for g in gestures_list:
                    # Create a group for the new gesture in the processed file
                    gesture_group = session_group.create_group(f'gesture_{g}')
                    
                    gesture_group.create_dataset('repetitions', data=np.array(repetitions_list))
                    # Iterate over repetitions in the current gesture
                    for r in repetitions_list:
                        # Apply preprocessing steps to the data
                        if processing != None:
                            emg_dabase = emg_preprocessing(h5_file[f'session_{s}'][f'gesture_{g}'][f'repetition_{r}']['emg'][:])
                        else:
                            emg_dabase = h5_file[f'session_{s}'][f'gesture_{g}'][f'repetition_{r}']['emg'][:]
                        # Create a group for the new repetition in the processed file
                        repetition_group = gesture_group.create_group(f'repetition_{r}')

                        # Save the preprocessed data in the new group
                        repetition_group.create_dataset('emg', data=emg_dabase)
                        

In [ ]:
h5_file['session_1'].keys()

In [ ]:
sessions=[1,2]
gestures=[0,1,2,3,4,5,6]
repetitions=[1,2,3,4,5,6,7,8,9,10]

emg_preprocessing = pp.ProcessingPipeline([
    pp.SampleTrimmer(axis=1, front=1024, back=1024),
    pp.Retification(),
    pp.EMGTKE(),
    pp.MAFilter(31)])

create_data(folder_path+'emg_file.hdf5', folder_path+'train/sub03.hdf5', [1], [1,3,5,7,9], gestures, processing=emg_preprocessing)
create_data(folder_path+'emg_file.hdf5', folder_path+'val/sub03.hdf5', [1], [10,4,6], gestures, processing=emg_preprocessing)
create_data(folder_path+'emg_file.hdf5', folder_path+'test/sub03.hdf5', [1], [8,2], gestures, processing=emg_preprocessing)
                #source, destination, session_list, repetitions_list, gestures_list, processing=None

In [ ]:
h5_file_train = h5py.File(folder_path+'train/sub03.hdf5', 'r')

In [ ]:
h5_file_train['session_1']['gesture_1'].keys()

In [ ]:
plt.plot(h5_file_train['session_1']['gesture_1']['repetition_3']['emg'][:].T)
plt.show()

In [ ]:
h5_file_train.keys()

In [ ]:
h5_file_train['session_1'].keys()

In [ ]:
h5_file_train['session_1']['gesture_0'].keys()

# Test the EMG extractor function

In [ ]:
h5_file['session_1'].keys()

In [ ]:
h5_file['session_1']['gesture_1'].keys()

In [ ]:
print('EMG signal shape',h5_file['session_1']['gesture_1']['repetition_1']['emg'][:].shape)

In [ ]:
plt.plot(h5_file['session_1']['gesture_1']['repetition_1']['emg'][:][4,:])

In [ ]:
real_movement=h5_file['session_1']['gesture_0']['repetition_6']['emg'][:]
real_movement.T.shape

In [ ]:
plt.plot(real_movement[params_ch,:].T)
plt.show()

In [ ]:
gesture=extract_emg_signal(real_movement,return_rest=True)
gesture.shape

In [ ]:
rest=extract_emg_signal(real_movement,return_rest=False)
rest.shape

In [ ]:
Movement=np.append(rest,gesture,axis=1)
Movement.shape

In [ ]:
th_m=np.zeros(Movement.shape[1]) 
th_m[rest.shape[1]:]=1

In [ ]:
fig,ax=plt.subplots(2,1, figsize=(10,8))
ax[0].plot(Movement[4:10].T)
ax[1].plot(th_m)
plt.show()


In [ ]:
h5_file['session_1']['gesture_0']['repetition_1']['emg'][0:16:].shape

In [ ]:
processed_emg.shape[0]

In [ ]:
processed_emg = extract_emg_signal(h5_file['session_1']['gesture_0']['repetition_1']['emg'][...], plot_result=True)

plt.figure(figsize=(15,3))
for ch in range(processed_emg.shape[0]):
    plt.plot(processed_emg[ch])
plt.show()

In [ ]:
processed_emg = extract_emg_signal(h5_file['session_1']['gesture_0']['repetition_1']['emg'][0:16:], return_rest=True, plot_result=True)

plt.figure(figsize=(15,3))
for ch in range(processed_emg.shape[0]):
    plt.plot(processed_emg[ch])
plt.show()

In [ ]:
processed_emg = extract_emg_signal(h5_file['session_1']['gesture_0']['repetition_1']['emg'][16:], plot_result=True)

plt.figure(figsize=(15,3))
for ch in range(processed_emg.shape[0]):
    plt.plot(processed_emg[ch])
plt.show()

In [ ]:
processed_emg = extract_emg_signal(h5_file['session_1']['gesture_0']['repetition_1']['emg'][16:], return_rest=True, plot_result=True)

plt.figure(figsize=(15,3))
for ch in range(processed_emg.shape[0]):
    plt.plot(processed_emg[ch])
plt.show()